# 🌍 Traducción Automática Neuronal (NMT) Wolof — NLLB-200 + ByT5

**Objetivo:** fine-tuning de un sistema de traducción neuronal para **wolof**
(lengua de bajo recurso) y lenguas vecinas, usando una arquitectura híbrida:

| Componente | Rol | Detalles |
|---|---|---|
| **ByT5-small** (`google/byt5-small`) | Modelo **entrenable** | Tokenización a nivel de bytes, ideal para lenguas sin tokenizador |
| **NLLB-200** (`facebook/nllb-200-distilled-600M`) | **Fallback** en inferencia | Traducción directa para pares bien cubiertos por Meta |

**Diseño:** solo **ByT5** se fine-tunea (NLLB es demasiado costoso de entrenar y
ya cubre muchos pares). En inferencia, si la lengua no está bien soportada por
NLLB o la confianza es baja, se usa la traducción de ByT5.

**Requisitos:**
- Python 3.10+, PyTorch, `transformers`, `datasets`, `sacrebleu`
- Datos paralelos en `DATA_DIR` (CSV, JSONL o pares de ficheros `.src`/`.tgt`)
- GPU recomendada (8 GB VRAM mínima); el notebook funciona en CPU con `byt5-small`

---
## 0. Instalación (solo la primera vez)

Ejecutar en terminal o descomentar en el notebook:

```bash
pip install torch transformers datasets sacrebleu tqdm
```

> ⚠️ **Modelos gated:** si el modelo requiere autenticación de Hugging Face,
> define la variable de entorno `HF_TOKEN` **antes** de ejecutar:
> `export HF_TOKEN=hf_...` (nunca se hardcodean tokens en el código).

In [ ]:
# === IMPORTS Y DISPOSITIVO ===
import os
import json
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    ByT5Tokenizer,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
# Token HF opcional (solo para modelos gated / push al Hub). NUNCA hardcodear.
print(f"HF_TOKEN definido: {bool(os.environ.get('HF_TOKEN'))}")

---
## 1. Configuración

Todo el notebook se configura desde `DATA_DIR` (ruta relativa o absoluta a los
datos) y el dataclass `Config`. No hay rutas hardcodeadas de Colab ni Windows.

In [ ]:
# === CONFIGURACIÓN ===
DATA_DIR = Path("./data")  # <- CAMBIA ESTO: ruta a tus datos paralelos
OUTPUT_DIR = Path("./models/nmt_hybrid")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
    # --- Modelos ---
    byt5_model_name: str = "google/byt5-small"          # Modelo entrenable (byte-level)
    nllb_model_name: str = "facebook/nllb-200-distilled-600M"  # Fallback en inferencia
    use_nllb_for_inference: bool = True                 # NLLB solo en inferencia (no entrenar)

    # --- Datos ---
    src_lang: str = "wo"       # ISO 639-1 (wolof)
    tgt_lang: str = "fr"       # ISO 639-1 (francés)
    max_length: int = 128
    max_train_samples: Optional[int] = None             # límite para pruebas rápidas

    # --- Entrenamiento ---
    num_epochs: int = 10
    batch_size: int = 8
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    gradient_accumulation_steps: int = 4
    warmup_steps: int = 100
    eval_every: int = 500          # pasos entre evaluaciones en val

    # --- Generación ---
    num_beams: int = 4
    seed: int = 42

config = Config()
print(config)

### Lenguas soportadas por NLLB-200

`SUPPORTED_LANGUAGES` mapea códigos ISO 639-1 a los códigos FLORES-200 que usa
NLLB (p. ej. `wo → wol_Latn`). Solo las lenguas de interés del proyecto se
mantienen aquí; añade más según necesites.

In [ ]:
# === MAPA DE LENGUAS (ISO 639-1 -> FLORES-200) ===
SUPPORTED_LANGUAGES = {
    "wo": "wol_Latn",   # Wolof  (Senegal, Gambia)
    "ff": "ful_Latn",   # Fula / Pulaar
    "bm": "bam_Latn",   # Bambara
    "srr": "srr_Latn",  # Serer
    "dyo": "dyo_Latn",  # Jola-Fonyi
    "snk": "snk_Latn",  # Soninké
    "es": "spa_Latn",   # Español
    "fr": "fra_Latn",   # Francés
    "it": "ita_Latn",   # Italiano
    "sw": "swh_Latn",   # Suajili
    "en": "eng_Latn",   # Inglés
    "pt": "por_Latn",   # Portugués
    "ar": "arb_Arab",   # Árabe
}
print(f"🌍 Idiomas soportados: {len(SUPPORTED_LANGUAGES)} -> {list(SUPPORTED_LANGUAGES)[:6]} ...")

---
## 2. Modelo híbrido NLLB + ByT5

`HybridNLLBByT5Model` envuelve:
- **ByT5**: modelo principal, único que se entrena (sus `forward` y `generate`).
- **NLLB**: cargado solo si `use_nllb_for_inference=True`; se usa únicamente
  como *fallback* en `nllb_translate()`.

Ambos son arquitecturas encoder-decoder de Hugging Face sobre PyTorch.

In [ ]:
# === MODELO HÍBRIDO (ByT5 entrenable + NLLB fallback) ===
class HybridNLLBByT5Model(nn.Module):
    """ByT5 para entrenamiento; NLLB-200 como respaldo en inferencia."""

    def __init__(self, config: Config):
        super().__init__()
        self.config = config

        # --- NLLB (solo inferencia) ---
        self.nllb_model, self.nllb_tokenizer = None, None
        if config.use_nllb_for_inference:
            try:
                print(f"🌍 Cargando NLLB ({config.nllb_model_name}) para inferencia...")
                self.nllb_model = AutoModelForSeq2SeqLM.from_pretrained(config.nllb_model_name)
                self.nllb_tokenizer = AutoTokenizer.from_pretrained(config.nllb_model_name)
                print("✅ NLLB cargado")
            except Exception as e:
                print(f"⚠️ No se pudo cargar NLLB (se usará solo ByT5): {e}")
                self.nllb_model, self.nllb_tokenizer = None, None

        # --- ByT5 (modelo principal, entrenable) ---
        print(f"🔤 Cargando ByT5 ({config.byt5_model_name})...")
        self.byt5_model = T5ForConditionalGeneration.from_pretrained(config.byt5_model_name)
        self.byt5_tokenizer = ByT5Tokenizer.from_pretrained(config.byt5_model_name)
        print(f"   Vocab ByT5: {self.byt5_tokenizer.vocab_size} | "
              f"pad={self.byt5_tokenizer.pad_token_id} eos={self.byt5_tokenizer.eos_token_id}")

    # ---- Entrenamiento: SIEMPRE ByT5 ----
    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        return self.byt5_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs,
        )

    def generate(self, *args, **kwargs):
        return self.byt5_model.generate(*args, **kwargs)

    # ---- Inferencia con NLLB (fallback) ----
    @torch.no_grad()
    def nllb_translate(self, texts: List[str], src: str, tgt: str,
                       max_length: Optional[int] = None) -> List[str]:
        """Traduce usando NLLB-200. Requiere lenguas en SUPPORTED_LANGUAGES."""
        if self.nllb_model is None:
            raise RuntimeError("NLLB no está cargado (use_nllb_for_inference=False)")
        src_code, tgt_code = SUPPORTED_LANGUAGES[src], SUPPORTED_LANGUAGES[tgt]
        self.nllb_tokenizer.src_lang = src_code
        self.nllb_model.config.forced_bos_token_id = (
            self.nllb_tokenizer.convert_tokens_to_ids(tgt_code)
        )
        enc = self.nllb_tokenizer(texts, return_tensors="pt", padding=True,
                                  truncation=True, max_length=max_length or 256).to(device)
        out = self.nllb_model.generate(
            **enc,
            max_new_tokens=max_length or 256,
            num_beams=self.config.num_beams,
        )
        return self.nllb_tokenizer.batch_decode(out, skip_special_tokens=True)

    def save(self, path: Path):
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        self.byt5_model.save_pretrained(path / "byt5")
        self.byt5_tokenizer.save_pretrained(path / "byt5")
        if self.nllb_model is not None:
            self.nllb_model.save_pretrained(path / "nllb")
            self.nllb_tokenizer.save_pretrained(path / "nllb")

model = HybridNLLBByT5Model(config).to(device)

---
## 3. Carga y tokenización de datos

Se esperan datos paralelos en `DATA_DIR` en uno de estos formatos:

1. **CSV** con columnas `source`, `target` (o `src`, `tgt`)
2. **JSONL** con claves `source` / `target`
3. **Pares de ficheros** `train.src` / `train.tgt` (una frase por línea)

La función `build_translation_pairs()` carga y devuelve una lista de pares
`(source, target)`. Después se tokenizan con el prefijo
`translate {src} to {tgt}: `, que es el formato que ByT5 aprende en fine-tuning.

In [ ]:
# === CARGA DE DATOS PARALELOS ===
def load_parallel_pairs(data_dir: Path, src_suffix: str = ".src",
                        tgt_suffix: str = ".tgt", max_samples: Optional[int] = None):
    """Carga pares (source, target) desde CSV / JSONL / ficheros .src/.tgt."""
    data_dir = Path(data_dir)
    pairs = []

    csvs = sorted(data_dir.glob("*.csv"))
    for csv_path in csvs:
        import pandas as pd
        df = pd.read_csv(csv_path)
        col_s, col_t = ("source", "target") if "source" in df.columns else ("src", "tgt")
        for _, row in df.iterrows():
            if isinstance(row[col_s], str) and isinstance(row[col_t], str):
                pairs.append((row[col_s].strip(), row[col_t].strip()))

    for jsonl_path in sorted(data_dir.glob("*.jsonl")):
        with open(jsonl_path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                s, t = obj.get("source", obj.get("src")), obj.get("target", obj.get("tgt"))
                if s and t:
                    pairs.append((str(s).strip(), str(t).strip()))

    src_files = sorted(data_dir.glob(f"*{src_suffix}"))
    for src_file in src_files:
        tgt_file = src_file.with_suffix(tgt_suffix)
        if not tgt_file.exists():
            continue
        with open(src_file, encoding="utf-8") as fs, open(tgt_file, encoding="utf-8") as ft:
            for s, t in zip(fs, ft):
                s, t = s.strip(), t.strip()
                if s and t:
                    pairs.append((s, t))

    # Deduplicar preservando orden
    pairs = list(dict.fromkeys(pairs))
    if max_samples:
        pairs = pairs[:max_samples]
    print(f"📊 Pares cargados: {len(pairs)}")
    if pairs:
        print(f"   Ejemplo: {pairs[0]}")
    return pairs


def tokenize_pairs(pairs, tokenizer, src: str, tgt: str, max_length: int):
    """Tokeniza pares con prefijo de instrucción 'translate X to Y: '."""
    prefix = f"translate {src} to {tgt}: "
    sources, targets = zip(*pairs)
    enc = tokenizer(
        [prefix + s for s in sources],
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    dec = tokenizer(
        list(targets),
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    # Labels: -100 donde hay padding (ignorado por la loss)
    labels = dec.input_ids.clone()
    labels[labels == tokenizer.pad_token_id] = -100
    return enc.input_ids, enc.attention_mask, labels


# --- Cargar y dividir ---
random.seed(config.seed)
torch.manual_seed(config.seed)

pairs = load_parallel_pairs(DATA_DIR, max_samples=config.max_train_samples)
random.shuffle(pairs)

n_val = max(1, int(len(pairs) * 0.05))
train_pairs, val_pairs = pairs[n_val:], pairs[:n_val]
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)}")

---
## 4. Dataset y DataLoader

`TranslationDataset` envuelve los pares ya tokenizados y `collate_fn` agrupa
los tensores en batches. `drop_last=True` evita batches incompletos que rompen
la acumulación de gradientes.

In [ ]:
# === DATASET Y DATALOADER ===
class TranslationDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


def make_dataloader(pairs, tokenizer, config: Config, shuffle: bool):
    input_ids, attn, labels = tokenize_pairs(pairs, tokenizer, config.src_lang,
                                             config.tgt_lang, config.max_length)
    ds = TranslationDataset(input_ids, attn, labels)
    return DataLoader(ds, batch_size=config.batch_size, shuffle=shuffle,
                      drop_last=shuffle)


train_loader = make_dataloader(train_pairs, model.byt5_tokenizer, config, shuffle=True)
val_loader = make_dataloader(val_pairs, model.byt5_tokenizer, config, shuffle=False)
print(f"Batches train: {len(train_loader)} | val: {len(val_loader)}")

---
## 5. Entrenamiento

Bucle de entrenamiento en **PyTorch puro**:
- Optimizador `AdamW` con `weight_decay`
- Scheduler lineal con warmup
- **AMP** (`torch.cuda.amp`) para acelerar en GPU con precisión mixta
- Acumulación de gradientes (`gradient_accumulation_steps`)
- Evaluación periódica en validación y guardado del mejor checkpoint

In [ ]:
# === ENTRENAMIENTO ===
def train_epoch(model, loader, optimizer, scheduler, scaler, config, epoch, global_step):
    model.train()
    total_loss, steps = 0.0, 0
    optimizer.zero_grad()

    for batch in tqdm(loader, desc=f"Epoch {epoch+1}/{config.num_epochs}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                            dtype=torch.float16, enabled=device.type == "cuda"):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / config.gradient_accumulation_steps

        scaler.scale(loss).backward()
        total_loss += loss.item() * config.gradient_accumulation_steps
        steps += 1
        global_step += 1

        if steps % config.gradient_accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if global_step % config.eval_every == 0:
            val_loss = evaluate(model, val_loader)
            print(f"   [step {global_step}] val_loss={val_loss:.4f}")

    return total_loss / max(steps, 1), global_step


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, count = 0.0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                            dtype=torch.float16, enabled=device.type == "cuda"):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total += outputs.loss.item()
        count += 1
    return total / max(count, 1)


optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate,
                              weight_decay=config.weight_decay)
total_steps = len(train_loader) * config.num_epochs // config.gradient_accumulation_steps
scheduler = get_linear_schedule_with_warmup(optimizer, config.warmup_steps, total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

best_val_loss = float("inf")
global_step = 0
for epoch in range(config.num_epochs):
    train_loss, global_step = train_epoch(
        model, train_loader, optimizer, scheduler, scaler, config, epoch, global_step)
    val_loss = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save(OUTPUT_DIR / "best")
        print(f"💾 Mejor checkpoint guardado en {OUTPUT_DIR / 'best'} (val_loss={val_loss:.4f})")

---
## 6. Inferencia

`translate()` usa **ByT5** con beam search. Si la lengua de destino está bien
cubierta por NLLB-200, `nllb_translate()` sirve como alternativa/fallback.

In [ ]:
# === INFERENCIA ===
@torch.no_grad()
def translate(model, texts, src=None, tgt=None, use_nllb_fallback=False):
    """Traduce con ByT5; opcionalmente con NLLB como respaldo."""
    if isinstance(texts, str):
        texts = [texts]
    src = src or config.src_lang
    tgt = tgt or config.tgt_lang

    if use_nllb_fallback and model.nllb_model is not None:
        try:
            return model.nllb_translate(texts, src, tgt, config.max_length)
        except Exception as e:
            print(f"⚠️ NLLB falló ({e}), usando ByT5")

    prefix = f"translate {src} to {tgt}: "
    enc = model.byt5_tokenizer(
        [prefix + t for t in texts], return_tensors="pt",
        padding=True, truncation=True, max_length=config.max_length,
    ).to(device)
    out = model.generate(
        **enc,
        max_new_tokens=config.max_length,
        num_beams=config.num_beams,
    )
    return model.byt5_tokenizer.batch_decode(out, skip_special_tokens=True)


# --- Prueba rápida ---
ejemplos = [
    "Jàmm nga am, naka mu taxa ci gis nga ma?",
    "Ndank ndank mooy gàntal",
    "Benn kër, benn askan, benn àdduna",
]
traducciones = translate(model, ejemplos, use_nllb_fallback=True)
for orig, trad in zip(ejemplos, traducciones):
    print(f"WO : {orig}")
    print(f"FR : {trad}")
    print()

---
## 7. Evaluación: BLEU y chrF

Se traduce el conjunto de validación con **beam search** y se comparan las
hipótesis con las referencias usando **sacrebleu** (BLEU y chrF), las métricas
estándar en NMT.

> ⚠️ Si no tienes GPU, reduce el subset evaluado (p. ej. `n_eval=200`) para
> que la evaluación termine en un tiempo razonable.

In [ ]:
# === EVALUACIÓN CON SACREBLEU (BLEU + chrF) ===
import sacrebleu

def evaluate_metrics(model, pairs, n_eval=None, src=None, tgt=None):
    """BLEU y chrF sobre un subset de pares (source -> target)."""
    src = src or config.src_lang
    tgt = tgt or config.tgt_lang
    if n_eval:
        pairs = pairs[:n_eval]

    sources, references = zip(*pairs)
    hypotheses = translate(model, list(sources), src, tgt, use_nllb_fallback=False)

    bleu = sacrebleu.corpus_bleu(hypotheses, [list(references)])
    chrf = sacrebleu.corpus_chrf(hypotheses, [list(references)])
    return bleu.score, chrf.score, hypotheses, references


bleu_score, chrf_score, hyps, refs = evaluate_metrics(model, val_pairs, n_eval=200)
print("=" * 50)
print(f"BLEU : {bleu_score:.2f}")
print(f"chrF : {chrf_score:.2f}")
print("=" * 50)
for h, r in list(zip(hyps, refs))[:5]:
    print(f"  REF : {r}")
    print(f"  HYP : {h}")
    print()

---
## 8. Guardar y publicar en Hugging Face Hub

El modelo y el tokenizador ByT5 se guardan localmente y, opcionalmente, se
suben al Hub. Para subir: define `HF_TOKEN` como variable de entorno
(`export HF_TOKEN=hf_...`) — **nunca** lo pegues en el notebook.

In [ ]:
# === GUARDAR MODELO ===
model.save(OUTPUT_DIR / "final")
print(f"✅ Modelo guardado en {OUTPUT_DIR / 'final'}")


# === PUBLICAR EN EL HUB (opcional) ===
def push_to_hub(repo_id: str):
    """Sube el modelo ByT5 fine-tuneado al Hub. Requiere HF_TOKEN en el entorno."""
    token = os.environ.get("HF_TOKEN")
    if not token:
        print("⚠️ HF_TOKEN no está definido. Ejecuta: export HF_TOKEN=hf_...")
        return
    model.byt5_model.push_to_hub(repo_id, token=token)
    model.byt5_tokenizer.push_to_hub(repo_id, token=token)
    print(f"🚀 Modelo publicado en https://huggingface.co/{repo_id}")


# push_to_hub("tu-usuario/nmt-wolof-byt5")  # <- descomenta para publicar